# 02 — Training

**What this notebook does:** trains the continuity-risk model on the company-year features built by `01_data_pipeline.ipynb`, evaluates it on a temporal holdout, saves the artifact, and optionally promotes it as the deployable model.

**The model.** Binary classifier predicting `continuity_risk_12m_label` — whether a company will stop being active (closed in INSEE, OR liquidation/radiation in BODACC, OR radiation/cessation in INPI) within 12 months after the feature cutoff date.

**The split.** Temporal: train on early years, test on the most recent year(s). No company appears in both train and test (any company in the test year contributes only its test-year row). This avoids the trap of evaluating on rows the model has already seen.

**The output.**

```
ml-artifacts/
├── model.joblib                          ← deployable model (overwritten each run)
├── model_metadata.json                   ← deployable model's metadata
├── model_run_comparison.csv              ← all runs' metrics, one row per run
├── model_run_comparison.png              ← comparison chart
└── runs/
    └── continuity-risk-YYYYMMDD-HHMMSS_<...>/
        ├── model.joblib                  ← archived copy of this run's artifact
        ├── metadata.json                 ← this run's metadata
        └── (per-run reports)
```

**Important:** each training run overwrites `ml-artifacts/model.joblib` (the deployable artifact). If you experiment with a new model family or hyperparameters and the result is *worse*, restore the previous one by copying from the corresponding `runs/<run_name>/model.joblib`. The Promote section near the bottom does this.

**Tweak freely.** Model family, year range, hyperparameters — all are exposed in the Configuration cell.

## Setup

Same setup as the data notebook — mount Drive, pull repo, install requirements, define paths.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'ml-workflow'
REPO_DIR = Path('/content/pfein')
BACKEND_DIR = REPO_DIR / 'back_end'

cwd = Path.cwd()
if (cwd / 'collabs' / 'requirements-colab.txt').exists():
    BACKEND_DIR = cwd
    REPO_DIR = BACKEND_DIR.parent

if not (BACKEND_DIR / 'collabs' / 'requirements-colab.txt').exists():
    if not (REPO_DIR / '.git').exists():
        subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)])
    else:
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin'])
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'switch', BRANCH])
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH])

DRIVE_ROOT = Path('/content/drive/MyDrive/PFE ML Data/pfe_data')
DATA_LAKE = DRIVE_ROOT / 'data-lake'
ARTIFACTS_DIR = DRIVE_ROOT / 'ml-artifacts'
DUCKDB_TMP = Path('/content/pfein_duckdb_tmp')
for p in (DATA_LAKE, ARTIFACTS_DIR, DUCKDB_TMP):
    p.mkdir(parents=True, exist_ok=True)
os.environ['DUCKDB_TEMP_DIRECTORY'] = str(DUCKDB_TMP)

os.chdir(BACKEND_DIR)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(BACKEND_DIR / 'collabs' / 'requirements-colab.txt')])

print(f'BACKEND_DIR   = {BACKEND_DIR}')
print(f'DATA_LAKE     = {DATA_LAKE}')
print(f'ARTIFACTS_DIR = {ARTIFACTS_DIR}')

## Configuration — what to train

**`MODEL_FAMILY`** — `'hgb'` (HistGradientBoosting from sklearn), `'lightgbm'`, `'xgboost'`, or `'catboost'`. Pick `hgb` for the default known-good baseline; the others require their respective packages installed.

**`TRAIN_START_YEAR` / `TRAIN_END_YEAR`** — which `prediction_year` rows to use for training (filtered before the temporal split). Set `TRAIN_END_YEAR` to the latest year you trust the *labels* for. Labels need a fully-observed 12-month forward window — so for training in May 2026, the safest `TRAIN_END_YEAR` is 2024 (since 2025 labels need observations through end of 2026 to be complete).

**`MAX_ROWS`** — set to e.g. `100_000` for a fast smoke test, `None` for full training.

**`MODEL_PARAMS`** — optional dict of hyperparameters. `None` uses the family's defaults from [app/tools/train_continuity_model.py](../../app/tools/train_continuity_model.py) (class-balanced, sensible regularization).

In [ ]:
MODEL_FAMILY     = 'hgb'                   # 'hgb' | 'lightgbm' | 'xgboost' | 'catboost'
TARGET           = 'continuity_risk_12m_label'

TRAIN_START_YEAR = 2017
TRAIN_END_YEAR   = 2024                    # see note above — leave at current_year - 2 for safe labels

MAX_ROWS         = None                    # None = full dataset, or e.g. 100_000 for smoke test
MIN_ROWS         = 1000                    # safety floor — refuses to train on too little data
GPU              = False                   # only used by lightgbm/xgboost/catboost

MODEL_PARAMS     = None                    # e.g. {'learning_rate': 0.05, 'max_depth': 6}

MODEL_FILE       = 'model.joblib'          # deployable artifact name

print(f'Training {MODEL_FAMILY} on {TARGET}')
print(f'Years {TRAIN_START_YEAR}-{TRAIN_END_YEAR}, max_rows={MAX_ROWS}')

## Train

**What `train_model` does internally** ([source](../../app/tools/train_continuity_model.py)):

1. Reads features from `data-lake/features/company_year_features/` and labels from `data-lake/features/risk_labels/`, joins them on `(siren, prediction_year)`.
2. Filters to `TRAIN_START_YEAR ≤ prediction_year ≤ TRAIN_END_YEAR`, drops rows with missing labels.
3. **Temporal split**: train on years up to `TRAIN_END_YEAR - 1`, test on `TRAIN_END_YEAR`. (Splits like this avoid leakage from future events into training.)
4. Wraps the model in a pipeline with a categorical-cardinality capper (so HistGradientBoosting can handle high-cardinality columns like activity_code).
5. Fits with class balancing (`class_weight='balanced'`) because the positive class is rare.
6. Computes metrics on the temporal test set: ROC-AUC, average precision, precision@0.5, recall@0.5, F1, threshold analysis, top-K capture analysis.
7. Saves the fitted pipeline + metadata as a *bundle* dict (`{'pipeline': ..., 'feature_columns': [...], ...}`) to `ml-artifacts/model.joblib` AND archives a copy under `ml-artifacts/runs/<run_name>/`.
8. Writes/updates `ml-artifacts/model_run_comparison.csv` with this run's metrics so you can compare runs over time.

**Smoke-test first.** If you've never run this on this data lake before, start with `MAX_ROWS = 100_000` and confirm the pipeline finishes end-to-end before launching a full training run.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(name)s | %(message)s', force=True)

from app.tools.train_continuity_model import train_model

train_model(
    data_lake_dir=DATA_LAKE,
    artifacts_dir=ARTIFACTS_DIR,
    model_file=MODEL_FILE,
    target=TARGET,
    min_rows=MIN_ROWS,
    max_rows=MAX_ROWS,
    train_start_year=TRAIN_START_YEAR,
    train_end_year=TRAIN_END_YEAR,
    model_family=MODEL_FAMILY,
    model_params=MODEL_PARAMS,
    gpu=GPU,
)
print('\nTraining done.')

## Inspect the new run's metadata

The metadata file `ml-artifacts/model_metadata.json` describes the currently-deployable model (the one just trained). Useful fields to scan:

- `model_version`, `run_name`, `model_family`
- `train_start_year`, `train_end_year`, `split_strategy`
- `metrics.average_precision`, `metrics.roc_auc` — primary quality signals for a rare-event ranking task
- `metrics.precision_at_0_5`, `metrics.recall_at_0_5`, `metrics.f1_at_0_5` — operating-point quality
- `feature_count`, `train_rows`, `test_rows`
- `class_counts`, `train_class_counts`, `test_class_counts` — to check class balance

In [ ]:
metadata = json.loads((ARTIFACTS_DIR / 'model_metadata.json').read_text(encoding='utf-8'))

print(f'Model version:   {metadata.get("model_version")}')
print(f'Run name:        {metadata.get("run_name")}')
print(f'Family:          {metadata.get("model_family")}')
print(f'Train years:     {metadata.get("train_start_year")} - {metadata.get("train_end_year")}')
print(f'Split strategy:  {metadata.get("split_strategy")}')
print(f'Feature count:   {metadata.get("feature_count")}')
print(f'Train rows:      {metadata.get("train_rows")}')
print(f'Test rows:       {metadata.get("test_rows")}')
print()
metrics = metadata.get('metrics', {})
print(f'ROC-AUC:             {metrics.get("roc_auc"):.4f}' if metrics.get('roc_auc') else 'ROC-AUC: n/a')
print(f'Average precision:   {metrics.get("average_precision"):.4f}' if metrics.get('average_precision') else 'AP: n/a')
print(f'Precision @ 0.5:     {metrics.get("precision_at_0_5"):.4f}' if metrics.get('precision_at_0_5') else 'P@0.5: n/a')
print(f'Recall @ 0.5:        {metrics.get("recall_at_0_5"):.4f}' if metrics.get('recall_at_0_5') else 'R@0.5: n/a')
print(f'F1 @ 0.5:            {metrics.get("f1_at_0_5"):.4f}' if metrics.get('f1_at_0_5') else 'F1@0.5: n/a')
print()
print(f'Class counts (train+test): {metadata.get("class_counts")}')
print(f'Train class counts:        {metadata.get("train_class_counts")}')
print(f'Test class counts:         {metadata.get("test_class_counts")}')

## Compare against previous runs

`ml-artifacts/model_run_comparison.csv` accumulates one row per training run over time, so you can see whether tweaking hyperparameters or switching model family actually helped. Sort by `average_precision` (the right primary metric for a rare-event problem) to find your best run.

In [ ]:
import pandas as pd
from IPython.display import display

comparison_path = ARTIFACTS_DIR / 'model_run_comparison.csv'
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    if not comparison.empty:
        comparison = comparison.sort_values('average_precision', ascending=False, na_position='last')
    columns_to_show = [c for c in [
        'model_version', 'run_name', 'model_family', 'split_strategy',
        'train_start_year', 'train_end_year',
        'average_precision', 'roc_auc',
        'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5',
        'trained_at',
    ] if c in comparison.columns]
    print(f'{len(comparison)} runs recorded. Top 10 by average_precision:')
    display(comparison[columns_to_show].head(10))
else:
    print('No model_run_comparison.csv yet (first training run, or the comparison was disabled).')

## Promote a specific run to deployable

By default, each training run overwrites `ml-artifacts/model.joblib`. If a *previous* run is better and you want to restore it as the deployable model, copy from its archived directory under `ml-artifacts/runs/<run_name>/`.

Find the `run_name` in the comparison table above (e.g. `continuity-risk-20260516-001351_continuity_hgb_full_dataset_900000`), set it below, and run the cell.

In [ ]:
RUN_TO_PROMOTE = None  # ← set e.g. 'continuity-risk-20260516-001351_continuity_hgb_full_dataset_900000'

if RUN_TO_PROMOTE:
    import shutil
    run_dir = ARTIFACTS_DIR / 'runs' / RUN_TO_PROMOTE
    if not run_dir.exists():
        raise FileNotFoundError(f'No archived run at {run_dir}')

    src_model = run_dir / 'model.joblib'
    src_meta  = run_dir / 'metadata.json'
    if not src_model.exists():
        raise FileNotFoundError(f'Run {RUN_TO_PROMOTE} has no model.joblib')

    # Back up the currently-deployable model before overwriting.
    current_model = ARTIFACTS_DIR / 'model.joblib'
    current_meta  = ARTIFACTS_DIR / 'model_metadata.json'
    if current_model.exists():
        backup_dir = ARTIFACTS_DIR / 'previous_deployable'
        backup_dir.mkdir(exist_ok=True)
        shutil.copy2(current_model, backup_dir / 'model.joblib')
        if current_meta.exists():
            shutil.copy2(current_meta, backup_dir / 'model_metadata.json')
        print(f'Backed up current deployable to {backup_dir}')

    shutil.copy2(src_model, current_model)
    if src_meta.exists():
        shutil.copy2(src_meta, current_meta)
    print(f'Promoted {RUN_TO_PROMOTE} → {current_model}')
else:
    print('No run selected (RUN_TO_PROMOTE is None). Leave as-is if the latest training is what you want deployed.')

## Done

If `average_precision` looks reasonable on the temporal test set and the run is the one you want deployed, you're set.

**Next:** `03_model_interrogation.ipynb` — load this artifact and score SIRENs interactively.

**Tuning ideas if metrics disappoint:**

- Run with a different `MODEL_FAMILY` (LightGBM and CatBoost often beat HGB on categorical-heavy data, given enough data).
- Pass `MODEL_PARAMS` with tuned hyperparameters — start with `{'learning_rate': 0.05, 'max_iter': 500}` for HGB.
- Extend `TRAIN_END_YEAR` to include more training data — but only if the labels for that year are fully observed (current year - 2 is the safe rule today).
- Add more features to the company-year table (see `01_data_pipeline.ipynb`'s notes on what raw data we currently don't mine — INPI nested formality fields, BODACC announcement text, INPI per-year balance sheets).